# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [64]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [65]:
# Load aviation accident data
df = pd.read_csv('AviationData.csv', encoding='latin1')

# Preview the first few rows of the main dataset, the aviation data
df.head()


C:\Users\Admin\AppData\Local\Temp\ipykernel_9592\1981143280.py:2: DtypeWarning: Columns (6,7,28) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('AviationData.csv', encoding='latin1')


,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,...,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007
3,20001218X45448,Accident,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000
4,20041105X01764,Accident,CHI79FA064,1979-08-02,"Canton, OH",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,1.0,2.0,NaN,0.0,VMC,Approach,Probable Cause,16-04-1980


In [66]:
# Inspect NaNs
print("\n***Inspecting NAN Values in the DataFrame***")
print(df.isna().sum().sort_values(ascending=False))


***Inspecting NAN Values in the DataFrame***
Schedule                  76307
Air.carrier               72241
FAR.Description           56866
Aircraft.Category         56602
Longitude                 54516
Latitude                  54507
Airport.Code              38757
Airport.Name              36185
Broad.phase.of.flight     27165
Publication.Date          13771
Total.Serious.Injuries    12510
Total.Minor.Injuries      11933
Total.Fatal.Injuries      11401
Engine.Type                7096
Report.Status              6384
Purpose.of.flight          6192
Number.of.Engines          6084
Total.Uninjured            5912
Weather.Condition          4492
Aircraft.damage            3194
Registration.Number        1382
Injury.Severity            1000
Country                     226
Amateur.Built               102
Model                        92
Make                         63
Location                     52
Investigation.Type            0
Event.Date                    0
Accident.Number           

In [68]:
# Inspect data types
print("\n***Inspecting Data Types in the DataFrame***")
print(df.dtypes)


***Inspecting Data Types in the DataFrame***
Event.Id                   object
Investigation.Type         object
Accident.Number            object
Event.Date                 object
Location                   object
Country                    object
Latitude                   object
Longitude                  object
Airport.Code               object
Airport.Name               object
Injury.Severity            object
Aircraft.damage            object
Aircraft.Category          object
Registration.Number        object
Make                       object
Model                      object
Amateur.Built              object
Number.of.Engines         float64
Engine.Type                object
FAR.Description            object
Schedule                   object
Purpose.of.flight          object
Air.carrier                object
Total.Fatal.Injuries      float64
Total.Serious.Injuries    float64
Total.Minor.Injuries      float64
Total.Uninjured           float64
Weather.Condition          object
Br

In [70]:
# Summary Statistics
print("\n***Summary Statistics for the DataFrame***")
print(df.describe())


***Summary Statistics for the DataFrame***
       Number.of.Engines  Total.Fatal.Injuries  Total.Serious.Injuries  \
count       82805.000000          77488.000000            76379.000000   
mean            1.146585              0.647855                0.279881   
std             0.446510              5.485960                1.544084   
min             0.000000              0.000000                0.000000   
25%             1.000000              0.000000                0.000000   
50%             1.000000              0.000000                0.000000   
75%             1.000000              0.000000                0.000000   
max             8.000000            349.000000              161.000000   

       Total.Minor.Injuries  Total.Uninjured  
count          76956.000000     82977.000000  
mean               0.357061         5.325440  
std                2.235625        27.913634  
min                0.000000         0.000000  
25%                0.000000         0.000000  
50%    

## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [71]:
# Rename columns for uniformity
df.columns = df.columns.str.lower().str.replace('.', '_', regex=False)
df.columns

Index(['event_id', 'investigation_type', 'accident_number', 'event_date',
       'location', 'country', 'latitude', 'longitude', 'airport_code',
       'airport_name', 'injury_severity', 'aircraft_damage',
       'aircraft_category', 'registration_number', 'make', 'model',
       'amateur_built', 'number_of_engines', 'engine_type', 'far_description',
       'schedule', 'purpose_of_flight', 'air_carrier', 'total_fatal_injuries',
       'total_serious_injuries', 'total_minor_injuries', 'total_uninjured',
       'weather_condition', 'broad_phase_of_flight', 'report_status',
       'publication_date'],
      dtype='object')

In [72]:
# Inspect relevant columns for analysis
print("\n***Filtering Relevant Columns for Analysis***")
filtered_df = ['event_date', 'location', 'country', 'latitude', 'longitude', 'airport_name', 
        'injury_severity', 'aircraft_damage', 'aircraft_category', 'make', 'model', 'amateur_built', 
        'number_of_engines', 'engine_type', 'purpose_of_flight', 'air_carrier', 'total_fatal_injuries', 
        'total_serious_injuries', 'total_minor_injuries', 'total_uninjured', 'weather_condition', 'broad_phase_of_flight', 'publication_date']
df[filtered_df].head()


***Filtering Relevant Columns for Analysis***


,event_date,location,country,latitude,longitude,airport_name,injury_severity,aircraft_damage,aircraft_category,make,...,engine_type,purpose_of_flight,air_carrier,total_fatal_injuries,total_serious_injuries,total_minor_injuries,total_uninjured,weather_condition,broad_phase_of_flight,publication_date
0,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,Fatal(2),Destroyed,NaN,Stinson,...,Reciprocating,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,NaN
1,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,Fatal(4),Destroyed,NaN,Piper,...,Reciprocating,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,19-09-1996
2,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,Fatal(3),Destroyed,NaN,Cessna,...,Reciprocating,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,26-02-2007
3,1977-06-19,"EUREKA, CA",United States,NaN,NaN,NaN,Fatal(2),Destroyed,NaN,Rockwell,...,Reciprocating,Personal,NaN,2.0,0.0,0.0,0.0,IMC,Cruise,12-09-2000
4,1979-08-02,"Canton, OH",United States,NaN,NaN,NaN,Fatal(1),Destroyed,NaN,Cessna,...,NaN,Personal,NaN,1.0,2.0,NaN,0.0,VMC,Approach,16-04-1980


In [74]:
# Filterered DataFrame with only relevant columns for analysis
df = df[filtered_df]

# Select only airplanes in the aircraft_category column
df = df[df['aircraft_category'] == 'Airplane']

# Remove Amateur Built aircrafts
df = df[df['amateur_built'] == 'No']

# Convert event_date and publication_date to datetime
df['event_date'] = pd.to_datetime(df['event_date'], errors='coerce')    
df['publication_date'] = pd.to_datetime(df['publication_date'], errors='coerce')

# Create year column from event_date for easier filtering
df['event_year'] = df['event_date'].dt.year

# Select events from 1986 onwards
df = df[df['event_year'] >= 1986]

# Verify Filtered DataFrame
print("\n***Preview of the Filtered DataFrame***")
print(df.shape)



***Preview of the Filtered DataFrame***
(21424, 24)


### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

**Cleaning Assumption for Injuries Columns**

Missing injury counts can be interpreted as: “No reported injuries of that type.”

Therefore, I shall fill the missing injuries values with 0.

Additionally, we will convert the data types from floats to integers as injuries are estimated by the number of people, which is to be in whole numbers. 



In [75]:
# Inspect injury columns
injury_df = df[['total_fatal_injuries', 'total_serious_injuries', 'total_minor_injuries', 'total_uninjured']]

# Cleaning: Fill NaN values in injury columns with 0
for col in injury_df.columns:
    df[col] = df[col].fillna(0)

print("\n***NaN Values in Injury Columns After Filling with 0***")
print(df[injury_df.columns].isna().sum())

# Convert injury columns to integers
for col in injury_df.columns:
    df[col] = df[col].astype(int)
print("\n***Data Types of Injury Columns After Conversion to Integer***")
print(df[injury_df.columns].dtypes)


***NaN Values in Injury Columns After Filling with 0***
total_fatal_injuries      0
total_serious_injuries    0
total_minor_injuries      0
total_uninjured           0
dtype: int64

***Data Types of Injury Columns After Conversion to Integer***
total_fatal_injuries      int64
total_serious_injuries    int64
total_minor_injuries      int64
total_uninjured           int64
dtype: object


**Derived Columns**


* > Total Occupants

Total occupants of each flight will be one of the the derived column as this will be the basis of analysing the rate of fatality and serious injuries rather than the actual count of fatality. This approach will be a fair comparison as it will give more meaningful analysis.

* > Fatality and Serious Injuries Rate

These two columns will provide the answers to the main client's needs; to identify aircrafts with low likelihood of fatal or serious injuries. 

In [77]:
# Estimate total people on board by summing injury columns
df['total_occupants'] = df['total_fatal_injuries'] + df['total_serious_injuries'] + df['total_minor_injuries'] + df['total_uninjured']

# Create the 'fatality_rate' column
df['fatality_rate'] = df['total_fatal_injuries'] / df['total_occupants']

# Create the 'serious_injury_rate' column
df['serious_injury_rate'] = df['total_serious_injuries'] / df['total_occupants']

print("\n***Check if new Columns Exists***")
df.columns


***Check if new Columns Exists***


Index(['event_date', 'location', 'country', 'latitude', 'longitude',
       'airport_name', 'injury_severity', 'aircraft_damage',
       'aircraft_category', 'make', 'model', 'amateur_built',
       'number_of_engines', 'engine_type', 'purpose_of_flight', 'air_carrier',
       'total_fatal_injuries', 'total_serious_injuries',
       'total_minor_injuries', 'total_uninjured', 'weather_condition',
       'broad_phase_of_flight', 'publication_date', 'event_year',
       'total_occupants', 'fatality_rate', 'serious_injury_rate'],
      dtype='object')

**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

Aircraft Damage measures aircraft robustness and destruction severity. Since it contains several null values, I will carefully inspect and clean it before analysis. 

I standardized text categories  using string cleaning operations to ensure consistency across labels

While the percentage of missing values for the column aircraft_damage is averagely 2.96%, this column is a core metric and poor imputation may lead to bias connclusions. Therefore, I shall remove the rows with missing values to avoid unreliable severity estimates. 

I will also create a binary metric for aircraft destruction rate to convert the severity of damage into a measurable probability as below; 

1 = aircraft destroyed
0 = aircraft not destroyed


In [78]:
# Inspect and clean Aircraft Damage columns

# Standardize text values in 'aircraft_damage' column
df['aircraft_damage'] = df['aircraft_damage'].str.strip().str.title()
print(df['aircraft_damage'].value_counts(dropna=False))

# Check unique values in 'aircraft_damage' column
print("\n***Unique Values in Aircraft Damage Column, include missing values***")
print(df['aircraft_damage'].value_counts(dropna=False))

# Check percentage of NaN values in 'aircraft_damage' column
print(f"\n***Percentage of NaN values in Aircraft Damage Column***")
print((df['aircraft_damage'].isna().mean()) * 100)

aircraft_damage
Substantial    16982
Destroyed       2308
NaN             1225
Minor            812
Unknown           97
Name: count, dtype: int64

***Unique Values in Aircraft Damage Column, include missing values***
aircraft_damage
Substantial    16982
Destroyed       2308
NaN             1225
Minor            812
Unknown           97
Name: count, dtype: int64

***Percentage of NaN values in Aircraft Damage Column***
5.717886482449589


In [79]:
# Drop rows with NaN values in 'aircraft_damage' column
df= df.dropna(subset=['aircraft_damage'])
df['aircraft_damage'].describe()

count           20199
unique              4
top       Substantial
freq            16982
Name: aircraft_damage, dtype: object

In [80]:
# Create a binary column 'destroyed' where 1 indicates the aircraft was destroyed and 0 indicates it was not
df['destroyed'] = np.where(
    df['aircraft_damage'] == 'Destroyed',
    1,
    0
)

df['destroyed'].value_counts(dropna=False)

destroyed
0    17891
1     2308
Name: count, dtype: int64

### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

The make column identifies the aircraft manufacturer and is central to the safety analysis since the client wants recommendations by aircraft make/model.

There is inconsistency in naming of the unique values in this column that will need cleaning.

Inconsistent capitalization, trailing whitespace, and naming inconsistencies will need standardization. 

Additionally, some records contain null values.
Rare manufacturers with insufficient observations
Manufacturers with very few accidents are not statistically reliable for comparison.
Cleaning Decisions

Cleaning decisions:
Remove leading/trailing spaces
Standardize capitalization
Standardize common manufacturer naming inconsistencies
Remove missing values
Keep only manufacturers with at least 50 accident records

This threshold improves statistical robustness and prevents misleading comparisons from extremely small samples.

In [82]:
# Check frequency of each unique value in 'make' column
make_df = df['make'].value_counts().reset_index()
make_df.columns = ['make', 'count']
make_df

,make,count
0,CESSNA,4786
1,PIPER,2775
2,Cessna,2251
3,Piper,1177
4,BEECH,1002
...,...,...
1310,T BIRD GOLDEN CIRCLE AIR INC,1
1311,TIGER AIRCRAFT LLC,1
1312,Beechcraft Corporation,1
1313,Hawker Beech,1


In [85]:
# Check missing values in 'make' column
df['make'].isna().sum()



np.int64(1)

In [86]:
# Inspect unique values in 'make' column
df['make'].nunique()

1315

In [87]:
# Drop rows with NaN values in 'make' column
df = df.dropna(subset=['make'])

# standardize text values in 'make' column
df['make'] = df['make'].str.strip().str.title()
df['make'] = df['make'].replace({'Boing': 'Boeing', 'Airbus Industrie': 'Airbus', 'Air Tractor': 'Air Tractor: Air Tractor Inc', 
                                 'Mcdonnell douglas': 'McDonnell Douglas', 'Lockheed': 'Lockheed Martin', 'Embraer S A': 'Embraer', 
                                 'Bombardier Inc': 'Bombardier', 'Cessna': 'Cessna Aircraft Company', 'Dehavilland': 'De Havilland', 
                                 'Dehavilland Aircraft': 'De Havilland', 'Piper': 'Piper Aircraft', 'Beech': 'Beechcraft', 
                                 'Gulfstream Aerospace Corporation': 'Gulfstream Aerospace', 'Dassault Aviation': 'Dassault', 
                                 'Fairchild': 'Fairchild Aircraft', 'Grumman': 'Grumman Aerospace', 'North American': 'North American Aviation', 
                                 'Cirrus': 'Cirrus Design Corp', 'Cirrus Design Corporation': 'Cirrus Design Corp',
                                 'Curtiss-Wright': 'Curtiss-Wright Corporation', 'Convair': 'Convair Corporation', 
                                 'Douglas': 'McDonnell Douglas', 'Learjet': 'Learjet Corporation'})

In [88]:
# Keep aircraft manufacturers with a threshold of 50 accident records
# Count accidents by make
make_counts = df['make'].value_counts()   

# Keep only makes with 50 or more accidents
makes_df = make_counts[make_counts >= 50].reset_index()

# Filter the main dataframe to include only the selected makes
df = df[df['make'].isin(makes_df['make'])]
df['make'].value_counts()

make
Cessna Aircraft Company           7048
Piper Aircraft                    3959
Beechcraft                        1422
Boeing                             678
Cirrus Design Corp                 360
Mooney                             359
Air Tractor Inc                    219
Bellanca                           218
Maule                              215
Air Tractor: Air Tractor Inc       206
Aeronca                            200
De Havilland                       157
Champion                           157
Grumman Aerospace                  145
Luscombe                           139
Stinson                            129
Airbus                             114
North American Aviation            104
Embraer                             98
Taylorcraft                         93
Aero Commander                      89
Aviat Aircraft Inc                  75
Diamond Aircraft Ind Inc            74
Socata                              73
Mcdonnell Douglas                   73
Bombardier          

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [90]:
# Inspect the column 'model' for missing values (Has 10 NaNs)
df['model'].isna().sum()

# Remove rows with NaN values in 'model' column
df = df.dropna(subset=['model'])
df['model'].isna().sum()

np.int64(0)

In [91]:
# Inspect column and counts for each make and model combination
make_model_counts = df.groupby(['make', 'model']).size().reset_index(name='count')
make_model_counts.sort_values(by='count', ascending=False).head(20)



,make,model,count
762,Cessna Aircraft Company,172,756
751,Cessna Aircraft Company,152,311
824,Cessna Aircraft Company,182,302
798,Cessna Aircraft Company,172S,270
1831,Piper Aircraft,PA28,268
793,Cessna Aircraft Company,172N,247
1158,Cirrus Design Corp,SR22,239
810,Cessna Aircraft Company,180,213
530,Boeing,737,194
792,Cessna Aircraft Company,172M,179


In [92]:
# Check if model labels are unique to each make.
# Note that some models may be shared across different makes.
# This can lead to duplicates when combining make and model into a single column. 
# This is important to check before creating a combined 'make_model' column.
make_model_counts['make_model'] = make_model_counts['make'] + '_' + make_model_counts['model']
make_model_counts[make_model_counts.duplicated(subset=['make_model'], keep=False)]
make_model_counts.head(20)

,make,model,count,make_model
0,Aero Commander,100,12,Aero Commander_100
1,Aero Commander,100 180,1,Aero Commander_100 180
2,Aero Commander,100-180,3,Aero Commander_100-180
3,Aero Commander,112,2,Aero Commander_112
4,Aero Commander,112A,1,Aero Commander_112A
5,Aero Commander,114B,1,Aero Commander_114B
6,Aero Commander,200,1,Aero Commander_200
7,Aero Commander,200D,3,Aero Commander_200D
8,Aero Commander,500,5,Aero Commander_500
9,Aero Commander,500 - B,1,Aero Commander_500 - B


In [93]:
# Create a derived column that is a unique identifier for a given plane type.
df['make_model'] = df['make'] + '_' + df['model']
df['make_model'].value_counts().head(20)




make_model
Cessna Aircraft Company_172     756
Cessna Aircraft Company_152     311
Cessna Aircraft Company_182     302
Cessna Aircraft Company_172S    270
Piper Aircraft_PA28             268
Cessna Aircraft Company_172N    247
Cirrus Design Corp_SR22         239
Cessna Aircraft Company_180     213
Boeing_737                      194
Cessna Aircraft Company_172M    179
Cessna Aircraft Company_150     175
Piper Aircraft_PA-18-150        174
Piper Aircraft_PA-28-140        165
Beechcraft_A36                  163
Cessna Aircraft Company_172P    142
Cessna Aircraft Company_140     115
Cessna Aircraft Company_172R    109
Cessna Aircraft Company_170B    107
Piper Aircraft_PA-28-180        105
Piper Aircraft_PA-28-161        102
Name: count, dtype: int64

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [97]:
# 1. Engine Type
# Check vale count (2626 NaNs)
df['engine_type'].value_counts(dropna=False)


# standardize formatting
df['engine_type'] = (
    df['engine_type']
    .str.strip()
    .str.title()
)

# Replace placeholder values and treat as NaN
df['engine_type'] = df['engine_type'].replace({
    'Unknown': np.nan,
    'Unk': np.nan
})

# inspect remaining categories
df['engine_type'].value_counts(dropna=False)


engine_type
Reciprocating      12786
NaN                 2714
Turbo Prop           899
Turbo Fan            413
Turbo Jet             48
Turbo Shaft            9
Geared Turbofan        1
Name: count, dtype: int64

In [98]:
# 2. Weather Condition
df['weather_condition'].value_counts(dropna=False)

# Standardize formatting
df['weather_condition'] = (
df['weather_condition'].str.strip()
    .str.upper()
) 

# Replace uknown values
df['weather_condition'] = df['weather_condition'].replace({
    'UNK': np.nan,
    'UNKNOWN': np.nan
})

# Verify cleaning
df['weather_condition'].value_counts(dropna=False)

weather_condition
VMC    14040
NaN     1967
IMC      863
Name: count, dtype: int64

In [99]:
# Number of Engines

# Numerical column. Inspect summary statistics
df['number_of_engines'].describe()

# Inspect unique values
df['number_of_engines'].value_counts(dropna=False)

# Convert data type to integer
df['number_of_engines'] = (
    df['number_of_engines']
    .astype('Int64')
)

df['number_of_engines'].head()


14357    2
14420    1
14421    1
14711    1
14712    1
Name: number_of_engines, dtype: Int64

In [100]:
# Purpose of Flight
df['purpose_of_flight'].value_counts(dropna=False)

# Standardize formatting
df['purpose_of_flight'] = (
    df['purpose_of_flight']
    .str.strip()
    .str.title()
)

# Replace uknowns
df['purpose_of_flight'] = df['purpose_of_flight'].replace({
    'Unknown': np.nan,
    'Unk': np.nan
})

# Group the rare_purpose category
purpose_counts = df['purpose_of_flight'].value_counts()

rare_purpose = purpose_counts[purpose_counts < 20].index

df['purpose_of_flight'] = df['purpose_of_flight'].replace(
    rare_purpose,
    'Other'
)

df['purpose_of_flight']

14357              Personal
14420    Aerial Application
14421    Aerial Application
14711             Skydiving
14712         Instructional
                ...        
88865         Instructional
88869                   NaN
88873              Personal
88877              Personal
88886              Personal
Name: purpose_of_flight, Length: 16870, dtype: object

In [101]:
# 5. BROAD PHASE OF FLIGHT
# Inspect the column
df['broad_phase_of_flight'].value_counts(dropna=False)

# Standardize formatting
df['broad_phase_of_flight'] = (
    df['broad_phase_of_flight']
    .str.strip()
    .str.title()
)

# Replace the uknown placeholders
df['broad_phase_of_flight'] = df[
    'broad_phase_of_flight'
].replace({
    'Unknown': np.nan,
    'Unk': np.nan
})

# Group rare phases
phase_counts = df['broad_phase_of_flight'].value_counts()

rare_phases = phase_counts[phase_counts < 20].index

df['broad_phase_of_flight'] = df[
    'broad_phase_of_flight'
].replace(
    rare_phases,
    'Other'
)

df['broad_phase_of_flight'].head()

14357        Landing
14420        Descent
14421       Standing
14711        Descent
14712    Maneuvering
Name: broad_phase_of_flight, dtype: object

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [102]:
# Inspect the dataframe

df.head()

,event_date,location,country,latitude,longitude,airport_name,injury_severity,aircraft_damage,aircraft_category,make,...,total_uninjured,weather_condition,broad_phase_of_flight,publication_date,event_year,total_occupants,fatality_rate,serious_injury_rate,destroyed,make_model
14357,1986-04-08,"MESA, AZ",United States,NaN,NaN,FALCON FIELD,Non-Fatal,Substantial,Airplane,Piper Aircraft,...,2,VMC,Landing,2011-08-03,1986,2,0.000000,0.0,0,Piper Aircraft_PA-23-250
14420,1986-04-15,"HANKAMER, TX",United States,NaN,NaN,NaN,Fatal(2),Destroyed,Airplane,Grumman Aerospace,...,0,VMC,Descent,2013-04-08,1986,2,1.000000,0.0,1,Grumman Aerospace_G164A
14421,1986-04-15,"HANKAMER, TX",United States,NaN,NaN,NaN,Fatal(2),Destroyed,Airplane,Grumman Aerospace,...,0,VMC,Standing,2013-04-08,1986,2,1.000000,0.0,1,Grumman Aerospace_G164A
14711,1986-05-19,"MEAD, WA",United States,NaN,NaN,NaN,Fatal(2),Destroyed,Airplane,Cessna Aircraft Company,...,0,VMC,Descent,2016-10-17,1986,3,0.666667,0.0,1,Cessna Aircraft Company_TU206F
14712,1986-05-19,"MEAD, WA",United States,NaN,NaN,NaN,Fatal(2),Destroyed,Airplane,Cessna Aircraft Company,...,0,VMC,Maneuvering,2016-10-17,1986,3,0.666667,0.0,1,Cessna Aircraft Company_152


In [ ]:
# Check the sum of Non_Null Values to select columns with 20,000 plus non-null values
df.notna().sum()

event_date                16870
location                  16867
country                   16869
latitude                  15580
longitude                 15576
airport_name              11435
injury_severity           16531
aircraft_damage           16870
aircraft_category         16870
make                      16870
model                     16870
amateur_built             16870
number_of_engines         15276
engine_type               14156
purpose_of_flight         14451
air_carrier                7778
total_fatal_injuries      16870
total_serious_injuries    16870
total_minor_injuries      16870
total_uninjured           16870
weather_condition         14903
broad_phase_of_flight      2407
publication_date          16226
event_year                16870
total_occupants           16870
fatality_rate             16485
serious_injury_rate       16485
destroyed                 16870
make_model                16870
dtype: int64

In [106]:
df.shape

(16870, 29)

In [108]:
#Apply a 70% data completeness to delete rows with a lot of null values
valid_columns = df.columns[
    df.notna().mean() > 0.7
]

df = df[valid_columns]
df.shape

(16870, 26)

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [109]:
df.to_csv('cleaned_aviation_data.csv', index=False)